<a href="https://colab.research.google.com/github/oit-cpai/cpai-foundations/blob/main/math/04_linear_systems_least_squares.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# M4 連立一次方程式と最小二乗法(notebook)

本文:[`04-linear-systems-least-squares.md`](https://github.com/oit-cpai/cpai-foundations/blob/main/math/04-linear-systems-least-squares.md)

**進め方**:予想 → 手計算 → コードで確かめる → 実装(`TODO`)→ 条件を変える → 説明。`check(...)` が `OK` と出るまで直してから次に進む。上のセルから順に実行すること。

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

def check(name, got, expected, tol=1e-8):
    got, expected = np.asarray(got, dtype=float), np.asarray(expected, dtype=float)
    if got.shape != expected.shape:
        print(f"NG  {name}: 形が違う  got {got.shape}, expected {expected.shape}")
    elif np.allclose(got, expected, atol=tol):
        print(f"OK  {name}")
    else:
        print(f"NG  {name}: got {got}, expected {expected}")

## 1. 2×2 の連立方程式

$\begin{pmatrix} 2 & 1 \\ 1 & -1 \end{pmatrix} x = \begin{pmatrix} 5 \\ 1 \end{pmatrix}$。

**手計算**:解を求める。

- 手計算:(ここに書く)

**実装**:本文の $A^{-1}$ の式を使って $2 \times 2$ の方程式を解く関数 `solve2x2` を書く。$\det A = 0$ のときは `None` を返す。

In [ ]:
def solve2x2(A, b):
    (a, bb), (c, d) = A
    det = a * d - bb * c
    if abs(det) < 1e-12:
        return None
    raise NotImplementedError("TODO: 本文の A^{-1} の式で x = A^{-1} b を計算する")

A = np.array([[2.0, 1.0], [1.0, -1.0]])
b = np.array([5.0, 1.0])
check("solve2x2", solve2x2(A, b), [2.0, 1.0])
check("np.linalg.solve と一致", solve2x2(A, b), np.linalg.solve(A, b))
check("代入すると b に戻る", A @ solve2x2(A, b), b)

2つの方程式を直線として描くと、解は交点になる。

In [ ]:
t = np.linspace(-1, 4, 100)
plt.figure(figsize=(4, 4))
plt.plot(t, 5 - 2 * t, label="2x1 + x2 = 5")
plt.plot(t, t - 1, label="x1 - x2 = 1")
x = np.linalg.solve(A, b)
plt.plot(*x, "ko"); plt.xlabel("x1"); plt.ylabel("x2"); plt.legend(); plt.grid(); plt.show()

## 2. 解がない・無数にあるとき:ランク

**予想**:次の2つの行列のランクはいくつか。`np.linalg.solve` はどうなるか。

- $A_1 = \begin{pmatrix} 1 & 2 \\ 2 & 4 \end{pmatrix}$、$A_2 = \begin{pmatrix} 1 & 2 \\ 2 & 4.001 \end{pmatrix}$
- 予想:(ここに書く)

In [ ]:
A1 = np.array([[1.0, 2.0], [2.0, 4.0]])
A2 = np.array([[1.0, 2.0], [2.0, 4.001]])
print("rank A1 =", np.linalg.matrix_rank(A1), "  rank A2 =", np.linalg.matrix_rank(A2))
print("solve2x2(A1, ...) =", solve2x2(A1, [3.0, 6.0]))
try:
    np.linalg.solve(A1, [3.0, 6.0])
except np.linalg.LinAlgError as e:
    print("np.linalg.solve(A1, ...) → エラー:", e)

# ほとんど平行なとき:b を少し変えると解が大きく変わる
print("A2 x = (3, 6)     →", np.round(np.linalg.solve(A2, [3.0, 6.0]), 3))
print("A2 x = (3, 6.001) →", np.round(np.linalg.solve(A2, [3.0, 6.001]), 3))

$b$ がわずか 0.001 変わっただけで、解が大きく変わる。行列式が 0 に近いときは、解が信用できないことがある。

## 3. 最小二乗法

3点 $(0, 1)$、$(1, 2)$、$(2, 2)$ に直線 $y = w_0 + w_1 t$ を当てはめる。$A$ は $3 \times 2$。

**実装**:正規方程式 $A^\top A\, w = A^\top b$ を `np.linalg.solve` で解く関数 `least_squares_normal` を書く。

In [ ]:
A3 = np.array([[1.0, 0.0], [1.0, 1.0], [1.0, 2.0]])
b3 = np.array([1.0, 2.0, 2.0])

def least_squares_normal(A, b):
    raise NotImplementedError("TODO: 正規方程式 A^T A w = A^T b を解く")

w = least_squares_normal(A3, b3)
check("手計算の答え (7/6, 1/2)", w, [7 / 6, 1 / 2])
check("np.linalg.lstsq と一致", w, np.linalg.lstsq(A3, b3, rcond=None)[0])

r = b3 - A3 @ w
print("残差 r =", np.round(r, 4))
check("残差は A のすべての列と直交する(A^T r = 0)", A3.T @ r, [0.0, 0.0])

tt = np.linspace(-0.5, 2.5, 50)
plt.figure(figsize=(4, 3))
plt.plot(A3[:, 1], b3, "o", label="data")
plt.plot(tt, w[0] + w[1] * tt, label="least squares")
for ti, bi, ri in zip(A3[:, 1], b3, r):
    plt.plot([ti, ti], [bi, bi - ri], "r:")
plt.legend(); plt.show()

赤い点線が残差である。最小二乗法は、この長さの2乗の合計を最も小さくする直線を選ぶ。

**最小になっていることを確かめる**:求めた $w$ を少しずらすと、残差の2乗和は必ず増えるはずである。

In [ ]:
def sse(w):
    return np.sum((b3 - A3 @ w) ** 2)

rng = np.random.default_rng(0)
increased = all(sse(w + 0.01 * rng.normal(size=2)) > sse(w) for _ in range(100))
print("最小二乗解での2乗和:", round(sse(w), 4), "  ずらすと必ず増えるか:", increased)

## 4. 強化学習の例:マルコフ連鎖の価値

$P = \begin{pmatrix} 0.9 & 0.1 \\ 0.5 & 0.5 \end{pmatrix}$、状態ごとの報酬 $r = (0, 1)^\top$、$\gamma = 0.9$。価値 $v$ は $(I - \gamma P)\, v = r$ を満たす。

**実装**:(1)`solve` で一度に解く関数と、(2)$v \leftarrow r + \gamma P v$ を何度もくり返す関数を書き、結果が一致することを確かめる。(2)は R4 の「反復方策評価」と同じ考え方である。

In [ ]:
P = np.array([[0.9, 0.1], [0.5, 0.5]])
r_s = np.array([0.0, 1.0])
gamma = 0.9

def value_by_solve(P, r, gamma):
    raise NotImplementedError("TODO: (I - gamma P) v = r を解く")

def value_by_iteration(P, r, gamma, n_iter):
    v = np.zeros(len(r))
    raise NotImplementedError("TODO: v <- r + gamma P v を n_iter 回くり返す")
    return v

v_exact = value_by_solve(P, r_s, gamma)
print("solve で求めた価値:", np.round(v_exact, 4))
check("価値の式を満たす", v_exact, r_s + gamma * P @ v_exact)
for n in [10, 50, 200]:
    v_it = value_by_iteration(P, r_s, gamma, n)
    print(f"反復 {n:3d} 回: {np.round(v_it, 4)}  誤差 {np.abs(v_it - v_exact).max():.2e}")

反復の回数を増やすと、`solve` の答えに近づく。誤差は1回ごとにおよそ $\gamma$ 倍になる。

## 5. 条件を変える

1. 4 で $\gamma = 0.99$ にすると、反復で同じ精度に達するのに何回かかるか。$\gamma = 1$ にすると `solve` はどうなるか(行列 $I - P$ のランクを調べる)
2. 3 で、点を1つ増やして(例:$(3, 4)$)最小二乗解を求め直す。残差が列と直交することは変わらないか
3. 2 の $A_2$ の (2,2) 成分を 4.001 から 4.1 に変えると、$b$ を変えたときの解の変化はどうなるか

## 6. 説明(自分の言葉で書く)

1. 方程式が未知数より多いとき、「解く」代わりに何をするのか
   - (ここに書く)
2. 最小二乗解が正しいかを、どう確かめられるか
   - (ここに書く)
3. 行列式が 0 に近い行列で方程式を解くと、何が困るか
   - (ここに書く)